# Semana 6 — NumPy I: fundamentos

**Programación para Ciencia de Datos**<br>
Sesiones: martes 29, miércoles 30 de septiembre y viernes 2 de octubre de 2026

> **De aquí sale la Práctica 3.** Esta semana y la siguiente son la base directa de la P3, donde vas a cargar tus columnas numéricas en arrays, calcular estadística vectorizada y **medir cuánto más rápido es NumPy que un bucle de Python**. El paso 1 de la P3 (cargar las columnas ignorando celdas vacías) y su benchmark los armamos completos el viernes, en el ejercicio integrador.

> **Lo que esta semana *no* cubre:** `np.median`, `np.percentile`, `np.std` por eje, máscaras booleanas para filtrar y `np.where` son de la **semana 7**. Aquí construimos el piso: qué es un array, cómo se crea, cómo se indexa y qué significa "vectorizado".

**Mapa de la semana**

| Sesión | Tema | Lo que vas a poder hacer al final |
|---|---|---|
| 1 (mar 29) | Arrays: creación, `dtype`, `shape`, indexación y *slicing* | Meter una columna de tu CSV en un array y sacarle pedazos |
| 2 (mié 30) | Operaciones vectorizadas, *broadcasting*, `ufuncs` | Calcular sobre 100,000 valores sin escribir un `for` |
| 3 (vie 2) | Ejercicio integrador | El paso 1 y el benchmark de la P3, funcionando |

## Sesión 1 — Arrays: el tipo de dato que lo cambia todo

### El problema que NumPy resuelve

En la P1 y la P2 procesaste ~103,000 filas con bucles de Python puro, y te diste cuenta de que tarda. No era tu código: era Python.

Antes de ver un solo concepto nuevo, mide el problema. La celda siguiente calcula el promedio de un millón de números de las dos formas.

In [1]:
import time
import numpy as np

# Un millon de numeros, primero como lista de Python y luego como array
lista = list(range(1_000_000))
arreglo = np.arange(1_000_000)

# Forma 1: bucle de Python puro
inicio = time.perf_counter()
total = 0.0
for v in lista:
    total += v
promedio_bucle = total / len(lista)
t_bucle = time.perf_counter() - inicio

# Forma 2: NumPy
inicio = time.perf_counter()
promedio_numpy = arreglo.mean()
t_numpy = time.perf_counter() - inicio

print("promedio con bucle:", promedio_bucle)
print("promedio con numpy:", promedio_numpy)
print()
print(f"bucle: {t_bucle:.4f} s")
print(f"numpy: {t_numpy:.4f} s")
print(f"numpy fue {t_bucle / t_numpy:.0f} veces mas rapido")

promedio con bucle: 499999.5
promedio con numpy: 499999.5

bucle: 0.0474 s
numpy: 0.0010 s
numpy fue 47 veces mas rapido


Mismo resultado, dos o tres órdenes de magnitud de diferencia. Y la versión de NumPy son **cuatro caracteres**: `.mean()`.

Eso es lo que compras con NumPy: menos código y mucha más velocidad. Lo que cuesta es aprender a pensar en **arrays completos** en vez de en elementos uno por uno. Eso es todo el tema de esta semana.

> **Los tiempos que ves aquí no son los que vas a ver tú.** Dependen de la máquina y de qué más esté corriendo. En la P3 tienes que reportar *tus* tiempos, medidos en tu equipo.

> **Para medir se usa `time.perf_counter()`, no `time.time()`.** Los dos se usan igual —restas dos lecturas y te da segundos—, pero en Windows `time.time()` solo avanza cada ~16 milisegundos, y NumPy es tan rápido que puede caber completo dentro de ese hueco: te reporta `0.0000 s`. Eso no solo se ve mal, además tumba el cálculo del factor de velocidad con un `ZeroDivisionError` al dividir entre cero. Volvemos a esto en el benchmark del viernes.

### Importar NumPy

Por convención universal se importa como `np`. No lo importes de otra forma: todo el código del mundo, incluida la documentación oficial, asume ese alias.

In [2]:
import numpy as np

print(np.__version__)

2.4.2


> **Si truena con `ModuleNotFoundError`:** te falta instalarlo en tu entorno virtual. Actívalo y corre `pip install numpy`, y después `pip freeze > requirements.txt` para que quede registrado en tu repo (acuérdate: ese archivo es parte del entregable del monorepo).

### Crear un array desde una lista

`np.array()` recibe una lista y devuelve un **ndarray** (*n-dimensional array*). Es el único tipo de dato nuevo de la semana; todo lo demás son cosas que se le hacen a un ndarray.

In [3]:
montos = np.array([850.0, 1200.0, 430.5, 2100.0, 675.0])

print(montos)
print(type(montos))

[ 850.  1200.   430.5 2100.   675. ]
<class 'numpy.ndarray'>


### Lista vs array: la diferencia que importa

Se parecen al imprimirse, pero no se comportan igual. Compara.

In [4]:
lista = [1, 2, 3, 4]
arreglo = np.array([1, 2, 3, 4])

print("lista  * 2 ->", lista * 2)      # repite la lista
print("array  * 2 ->", arreglo * 2)    # multiplica cada elemento
print()
print("lista  + lista ->", lista + lista)        # concatena
print("array  + array ->", arreglo + arreglo)    # suma elemento a elemento

lista  * 2 -> [1, 2, 3, 4, 1, 2, 3, 4]
array  * 2 -> [2 4 6 8]

lista  + lista -> [1, 2, 3, 4, 1, 2, 3, 4]
array  + array -> [2 4 6 8]


Esa es la idea central: **en un array, las operaciones se aplican a cada elemento**. Se le llama *vectorización* y es el tema de la sesión 2.

| | Lista de Python | Array de NumPy |
|---|---|---|
| Tipos de sus elementos | Mezclados (`[1, "a", None]`) | **Uno solo** para todo el array |
| `* 2` | Repite la lista | Multiplica cada elemento |
| `+` | Concatena | Suma elemento a elemento |
| Tamaño | Crece con `.append()` | **Fijo** al crearse |
| Operaciones matemáticas | A mano, con bucle | Integradas y vectorizadas |
| Velocidad en cálculo | Lenta | Rápida |
| Memoria | Más (cada número es un objeto) | Menos (bloque compacto) |

### Un array es homogéneo: el `dtype`

Todos los elementos de un array comparten un tipo, el `dtype`. NumPy lo deduce al crear el array, y si le pasas tipos mezclados **no falla: convierte todo al tipo más general**. Eso es lo que hay que saber ver.

In [5]:
print(np.array([1, 2, 3]).dtype)          # enteros
print(np.array([1, 2.5, 3]).dtype)        # un float contagia a todos
print(np.array([1, 2, True]).dtype)       # True vale 1
print(np.array([1, 2, "3"]).dtype)        # un string convierte TODO a texto

int64
float64
int64
<U21


Ese último caso es la trampa que te va a morder con tu CSV. Mira qué pasa:

In [6]:
# Esto es lo que sale de leer una columna del CSV con .split("|")
crudo = np.array(["850.0", "1200.0", "", "2100.0"])

print("dtype:", crudo.dtype)    # <U6 = texto Unicode de hasta 6 caracteres
print(crudo)
print()
print("Y por eso esto NO suma, concatena:")
print(crudo[0] + crudo[1])

dtype: <U6
['850.0' '1200.0' '' '2100.0']

Y por eso esto NO suma, concatena:
850.01200.0


`<U6` significa "texto Unicode de hasta 6 caracteres". Si lees tu CSV y metes los valores directo a `np.array()` **sin convertirlos**, obtienes un array de texto y ninguna operación matemática funciona. Convertir a número es el paso 1 de la P3 y lo hacemos completo el viernes.

### Convertir tipos con `astype`

`astype` devuelve un **array nuevo** con otro `dtype` (no modifica el original).

In [7]:
textos = np.array(["850.0", "1200.0", "430.5"])
numeros = textos.astype(float)

print("original:", textos, textos.dtype)
print("convertido:", numeros, numeros.dtype)
print("suma:", numeros.sum())

original: ['850.0' '1200.0' '430.5'] <U6
convertido: [ 850.  1200.   430.5] float64
suma: 2480.5


Dos avisos sobre `astype`:

1. **De float a int trunca, no redondea.** `2.9` se vuelve `2`. Si quieres redondear, usa `np.round()` *antes*.
2. **Si un solo valor no es convertible, truena todo el array.** Y tu columna trae celdas vacías y basura como `"abc"`, así que esto te va a pasar.

In [8]:
# 1. Truncamiento
print(np.array([2.9, -2.9, 3.5]).astype(int))
print(np.round(np.array([2.9, -2.9, 3.5])).astype(int))

# 2. Un valor malo impide la conversion completa
sucio = np.array(["850.0", "", "abc", "2100.0"])
try:
    sucio.astype(float)
except ValueError as e:
    print()
    print("ValueError:", e)

[ 2 -2  3]
[ 3 -3  4]

ValueError: could not convert string to float: np.str_('')


> **Guárdate ese error.** Resolverlo —convertir lo que se pueda y marcar el resto— es exactamente el paso 2 del ejercicio integrador del viernes y del paso 1 de la P3.

### Crear arrays sin escribir los datos a mano

Cuatro funciones que vas a usar todo el tiempo: `arange`, `linspace`, `zeros` y `ones`.

In [9]:
# arange: como range(), pero devuelve un array y acepta pasos decimales
print(np.arange(5))
print(np.arange(2, 10))
print(np.arange(0, 10, 2))
print(np.arange(1.0, 2.0, 0.25))

[0 1 2 3 4]
[2 3 4 5 6 7 8 9]
[0 2 4 6 8]
[1.   1.25 1.5  1.75]


In [10]:
# linspace: N valores repartidos entre dos limites, AMBOS incluidos
print(np.linspace(0, 1, 5))
print(np.linspace(0, 100, 3))

[0.   0.25 0.5  0.75 1.  ]
[  0.  50. 100.]


In [11]:
# zeros / ones / full: arrays de relleno, muy utiles para "reservar lugar"
print(np.zeros(5))
print(np.ones(3))
print(np.full(4, 7.5))
print()
# Tambien en 2-D, pasando una tupla (filas, columnas)
print(np.zeros((2, 3)))

[0. 0. 0. 0. 0.]
[1. 1. 1.]
[7.5 7.5 7.5 7.5]

[[0. 0. 0.]
 [0. 0. 0.]]


In [12]:
cubo = np.ones((3,4,5))
print(cubo.shape)
print(cubo.ndim)
print(cubo.size)
print(cubo)
print(cubo.dtype)

(3, 4, 5)
3
60
[[[1. 1. 1. 1. 1.]
  [1. 1. 1. 1. 1.]
  [1. 1. 1. 1. 1.]
  [1. 1. 1. 1. 1.]]

 [[1. 1. 1. 1. 1.]
  [1. 1. 1. 1. 1.]
  [1. 1. 1. 1. 1.]
  [1. 1. 1. 1. 1.]]

 [[1. 1. 1. 1. 1.]
  [1. 1. 1. 1. 1.]
  [1. 1. 1. 1. 1.]
  [1. 1. 1. 1. 1.]]]
float64


**`arange` o `linspace`?** `arange` cuando lo que te importa es el **paso** (de 0 a 10 de 2 en 2); `linspace` cuando lo que te importa es **cuántos valores** quieres (10 puntos entre 0 y 1).

Y una advertencia sobre `arange` con pasos decimales: los flotantes no son exactos, así que el número de elementos puede sorprenderte.

In [13]:
a = np.arange(0, 1, 0.1)
print(a)
print("cuantos elementos:", len(a))      # 10, el 1.0 no entra
print()
b = np.linspace(0, 1, 11)
print(b)
print("cuantos elementos:", len(b))      # 11, el 1.0 si entra

[0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
cuantos elementos: 10

[0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
cuantos elementos: 11


### Los cuatro atributos que siempre revisas

Cuando algo no funciona con NumPy, el 80% de las veces es porque el array no tiene la forma o el tipo que creías. Estos cuatro atributos son tu diagnóstico.

In [14]:
montos = np.array([850.0, 1200.0, 430.5, 2100.0, 675.0])

print("array :", montos)
print("shape :", montos.shape)    # la forma, SIEMPRE una tupla
print("ndim  :", montos.ndim)     # cuantas dimensiones
print("size  :", montos.size)     # cuantos elementos en total
print("dtype :", montos.dtype)    # de que tipo son

array : [ 850.  1200.   430.5 2100.   675. ]
shape : (5,)
ndim  : 1
size  : 5
dtype : float64


> **`shape` siempre es una tupla.** Para un array de 1-D con 5 elementos es `(5,)` —con esa coma suelta—, no `5`. La coma está ahí porque es una tupla de un solo elemento.

### Arrays de 2-D: una tabla

Un array de 2 dimensiones es lo que ya conoces como tabla: filas y columnas. Tu CSV es naturalmente 2-D, y en la P3 vas a juntar tus dos columnas numéricas en una matriz así.

Se crea desde una **lista de listas**, donde cada lista interna es una fila.

In [15]:
# 3 registros x 2 columnas: [monto_venta, minutos_en_sitio]
tabla = np.array([
    [850.0, 12.5],
    [1200.0, 33.0],
    [430.5, 8.0],
])

print(tabla)
print()
print("shape:", tabla.shape)     # (filas, columnas)
print("ndim :", tabla.ndim)
print("size :", tabla.size)      # 3 * 2 = 6

[[ 850.    12.5]
 [1200.    33. ]
 [ 430.5    8. ]]

shape: (3, 2)
ndim : 2
size : 6


`shape` se lee siempre **(filas, columnas)**. `(3, 2)` son 3 filas y 2 columnas. Si lo lees al revés, todos tus `axis` de la semana que entra van a salir mal.

### `reshape`: la misma memoria con otra forma

`reshape` reorganiza los elementos en otra forma, siempre que el total cuadre.

In [16]:
a = np.arange(12)
print(a)
print()
print(a.reshape(3, 4))
print()
print(a.reshape(4, 3))
print()
# -1 significa "calculalo tu": quiero 2 columnas, las filas que salgan
print(a.reshape(-1, 2))

[ 0  1  2  3  4  5  6  7  8  9 10 11]

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]

[[ 0  1]
 [ 2  3]
 [ 4  5]
 [ 6  7]
 [ 8  9]
 [10 11]]


In [17]:
# Si el total no cuadra, truena: 12 elementos no caben en 5x3
try:
    np.arange(12).reshape(5, 3)
except ValueError as e:
    print("ValueError:", e)

ValueError: cannot reshape array of size 12 into shape (5,3)


### Indexación y *slicing* en 1-D

Igual que en las listas de Python: desde 0, índices negativos desde el final, y rebanadas `[inicio:fin:paso]` donde `fin` **no** se incluye.

In [18]:
a = np.array([10, 20, 30, 40, 50, 60])

print("a       :", a)
print("a[0]    :", a[0])        # primero
print("a[-1]   :", a[-1])       # ultimo
print("a[1:4]  :", a[1:4])      # del 1 al 3 (el 4 NO entra)
print("a[:3]   :", a[:3])       # los primeros 3
print("a[3:]   :", a[3:])       # del 3 al final
print("a[::2]  :", a[::2])      # de 2 en 2
print("a[::-1] :", a[::-1])     # al reves
print("a[-2::-3] :", a[-2::-3])

a       : [10 20 30 40 50 60]
a[0]    : 10
a[-1]   : 60
a[1:4]  : [20 30 40]
a[:3]   : [10 20 30]
a[3:]   : [40 50 60]
a[::2]  : [10 30 50]
a[::-1] : [60 50 40 30 20 10]
a[-2::-3] : [50 20]


### *Slicing* en 2-D: `[fila, columna]`

Aquí sí cambia respecto a las listas. En una lista de listas tendrías que escribir `tabla[1][0]`; en NumPy se usa **un solo corchete con coma**: `tabla[1, 0]`. Y cada posición acepta una rebanada.

In [19]:
tabla = np.arange(1, 13).reshape(3, 4)
print(tabla)
print()
print("tabla[1, 2]   ->", tabla[1, 2])      # fila 1, columna 2: un solo valor
print("tabla[0]      ->", tabla[0])         # toda la fila 0
print("tabla[:, 1]   ->", tabla[:, 1])      # toda la columna 1
print("tabla[-1]     ->", tabla[-1])        # ultima fila
print("tabla[:, -1]  ->", tabla[:, -1])     # ultima columna

print("tabla[::2, ::-2] ->", tabla[::2, ::-2])

print("tabla[-1::-3, -1::-3] ->", tabla[-1::-3, -1::-3])

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

tabla[1, 2]   -> 7
tabla[0]      -> [1 2 3 4]
tabla[:, 1]   -> [ 2  6 10]
tabla[-1]     -> [ 9 10 11 12]
tabla[:, -1]  -> [ 4  8 12]
tabla[::2, ::-2] -> [[ 4  2]
 [12 10]]
tabla[-1::-3, -1::-3] -> [[12  9]]


In [20]:
# Submatrices: una rebanada en cada posicion
print("filas 0-1, columnas 1-2:")
print(tabla[0:2, 1:3])
print()
print("todas las filas, las dos primeras columnas:")
print(tabla[:, 0:2])

filas 0-1, columnas 1-2:
[[2 3]
 [6 7]]

todas las filas, las dos primeras columnas:
[[ 1  2]
 [ 5  6]
 [ 9 10]]


Los patrones que vas a repetir mil veces:

| Lo que quieres | Cómo se escribe |
|---|---|
| Un valor | `tabla[fila, columna]` |
| Una fila completa | `tabla[2]` o `tabla[2, :]` |
| **Una columna completa** | `tabla[:, 1]` |
| Las primeras 5 filas | `tabla[:5]` |
| Un bloque | `tabla[0:3, 1:4]` |
| La última columna | `tabla[:, -1]` |

El que más vas a usar es `tabla[:, i]`: saca la columna *i*. Ahí es donde viven tus variables numéricas.

### La trampa más importante de la semana: las rebanadas son *vistas*

En Python, `lista[0:3]` te da una **copia**. En NumPy, `array[0:3]` te da una **vista**: una ventana a la misma memoria. Si la modificas, modificas el original.

In [21]:
original = np.array([10, 20, 30, 40, 50])
pedazo = original[0:3]

pedazo[0] = 999          # modifico el pedazo...

print("pedazo  :", pedazo)
print("original:", original)     # ...y el original cambio tambien
print()
print("comparten memoria?", np.shares_memory(original, pedazo))

pedazo  : [999  20  30]
original: [999  20  30  40  50]

comparten memoria? True


No es un error de NumPy, es la razón por la que es rápido: no copia 100,000 valores cada vez que tomas una rebanada. Pero te explota en la cara si no lo sabes.

Cuando necesites un pedazo independiente, pide la copia explícitamente con `.copy()`.

In [22]:
original = np.array([10, 20, 30, 40, 50])
copia = original[0:3].copy()

copia[0] = 999

print("copia   :", copia)
print("original:", original)     # intacto
print("comparten memoria?", np.shares_memory(original, copia))

copia   : [999  20  30]
original: [10 20 30 40 50]
comparten memoria? False


> **Regla práctica:** si vas a modificar un pedazo y quieres conservar el original, `.copy()`. En la P3 y la P5, cuando empieces a corregir valores de una columna, esta es la diferencia entre limpiar tus datos y arruinarlos sin darte cuenta.

### Indexación con lista de índices

Además de rebanadas, puedes pedir posiciones sueltas pasando una lista. Esto sí devuelve una **copia** (no una vista).

In [23]:
a = np.array([10, 20, 30, 40, 50, 60])

print(a[[0, 2, 4]])        # posiciones 0, 2 y 4
print(a[[-1, -2]])         # las dos ultimas, en ese orden
print()
print("es copia?", not np.shares_memory(a, a[[0, 2, 4]]))

[10 30 50]
[60 50]

es copia? True


### Ejercicios sesión 1

Resuélvelos en estas celdas. Si te atoras, lo que necesitas está arriba.

In [24]:
# Ejercicio 1: Crea un array con los numeros 5, 10, 15, 20, 25 de DOS formas
# distintas: escribiendolos a mano y usando np.arange.
# Imprime los dos y verifica que son iguales.

# Tu codigo aqui:

In [25]:
# Ejercicio 2: Crea un array con 7 valores igualmente repartidos entre 0 y 300
# (los dos extremos incluidos). Imprime su shape, su size y su dtype.

# Tu codigo aqui:

In [26]:
# Ejercicio 3: Este array viene de leer una columna del CSV:
columna = np.array(["12.5", "33.0", "8.75", "41.0"])
# a) Imprime su dtype y explica en un comentario por que no puedes sumarlo.
# b) Conviertelo a float y calcula su suma.

# Tu codigo aqui:

In [27]:
# Ejercicio 4: Construye una matriz 4x3 con los numeros del 1 al 12.
# Luego imprime, usando slicing:
#   a) la segunda fila
#   b) la ultima columna
#   c) el bloque de las dos primeras filas y las dos ultimas columnas
#   d) el valor que esta en la fila 3, columna 1

# Tu codigo aqui:

In [28]:
# Ejercicio 5: Crea un array con los numeros del 100 al 109.
# a) Toma una rebanada con los 4 primeros y cambiale el primer valor a 0.
# b) Imprime el array original. Que paso y por que?
# c) Repitelo de forma que el original NO se vea afectado.

# Tu codigo aqui:

In [29]:
# Ejercicio 6 (reto): np.zeros(5) crea 5 ceros flotantes.
# Averigua (probando, o con la ayuda de ?) como crear 5 ceros ENTEROS,
# y comprueba el dtype de los dos.
# Pista: zeros acepta un segundo argumento.

# Tu codigo aqui:

## Sesión 2 — Operaciones vectorizadas

Ayer aprendiste a crear arrays y a sacarles pedazos. Hoy viene la parte por la que NumPy existe: **operar sobre todos los elementos a la vez, sin escribir un `for`**.

### Aritmética con un escalar

Cualquier operación entre un array y un número se aplica a cada elemento.

In [30]:
montos = np.array([850.0, 1200.0, 430.5, 2100.0])

print("original :", montos)
print("+ 100    :", montos + 100)
print("* 2      :", montos * 2)
print("/ 1000   :", montos / 1000)
print("** 0.5   :", montos ** 0.5)       # raiz cuadrada
print("// 100   :", montos // 100)       # division entera
print("% 100    :", montos % 100)        # residuo

original : [ 850.  1200.   430.5 2100. ]
+ 100    : [ 950.  1300.   530.5 2200. ]
* 2      : [1700. 2400.  861. 4200.]
/ 1000   : [0.85   1.2    0.4305 2.1   ]
** 0.5   : [29.15475947 34.64101615 20.74849392 45.82575695]
// 100   : [ 8. 12.  4. 21.]
% 100    : [50.   0.  30.5  0. ]


Nota que el original nunca cambia: cada operación devuelve un **array nuevo**. Si quieres guardar el resultado, asígnalo a una variable.

### Operaciones entre dos arrays

Si los dos arrays tienen la misma forma, la operación va **posición por posición**.

In [31]:
monto = np.array([850.0, 1200.0, 430.5, 2100.0])
minutos = np.array([12.5, 33.0, 8.0, 51.0])

print("suma      :", monto + minutos)
print("producto  :", monto * minutos)
print()
# El caso util de verdad: una metrica derivada, en una linea
print("pesos por minuto:", np.round(monto / minutos, 2))

suma      : [ 862.5 1233.   438.5 2151. ]
producto  : [ 10625.  39600.   3444. 107100.]

pesos por minuto: [68.   36.36 53.81 41.18]


> **`*` multiplica elemento a elemento, no es producto de matrices.** Para producto matricial se usa `@` o `np.dot`. En este curso no lo necesitas, pero si lo ves en código ajeno, ya sabes que no es lo mismo.

### El bucle que ya no vas a escribir

Compara las dos versiones del mismo cálculo. Fíjate en el número de líneas y en el número de cosas que pueden salir mal en cada una.

In [32]:
monto = [850.0, 1200.0, 430.5, 2100.0]
minutos = [12.5, 33.0, 8.0, 51.0]

# Version Python puro
resultado_bucle = []
for i in range(len(monto)):
    resultado_bucle.append(round(monto[i] / minutos[i], 2))

# Version NumPy
resultado_numpy = np.round(np.array(monto) / np.array(minutos), 2)

print("bucle:", resultado_bucle)
print("numpy:", resultado_numpy)
print("iguales?", np.allclose(resultado_bucle, resultado_numpy))

bucle: [68.0, 36.36, 53.81, 41.18]
numpy: [68.   36.36 53.81 41.18]
iguales? True


Con 4 valores da lo mismo. Con 103,000 valores, la versión de NumPy sigue siendo una línea y tarda milisegundos.

> **`np.allclose`** es la forma correcta de comparar arrays de flotantes: verifica que sean iguales *dentro de una tolerancia*. Nunca compares flotantes con `==`.

### *Broadcasting*: cuando las formas no coinciden

Ya lo usaste sin saberlo: `montos + 100` funciona aunque `montos` tenga 4 elementos y `100` sea uno solo. NumPy **estiró** el escalar para que cuadrara. Eso es *broadcasting*.

La regla general: NumPy compara las formas **de derecha a izquierda**, y dos dimensiones son compatibles si son iguales o si una de ellas es 1.

In [33]:
tabla = np.array([
    [100.0, 200.0, 300.0],
    [400.0, 500.0, 600.0],
])
print("tabla, shape", tabla.shape)
print(tabla)

# (2,3) con un escalar
print()
print("tabla * 2:")
print(tabla * 2)

tabla, shape (2, 3)
[[100. 200. 300.]
 [400. 500. 600.]]

tabla * 2:
[[ 200.  400.  600.]
 [ 800. 1000. 1200.]]


In [34]:
# (2,3) con un vector de (3,): el vector se aplica a CADA fila
ajuste = np.array([1.0, 10.0, 100.0])

print("tabla shape :", tabla.shape)
print("ajuste shape:", ajuste.shape)
print()
print(tabla * ajuste)

tabla shape : (2, 3)
ajuste shape: (3,)

[[  100.  2000. 30000.]
 [  400.  5000. 60000.]]


In [35]:
# (3,1) con (1,4): se estiran las DOS y sale una matriz (3,4)
columna = np.array([[1], [2], [3]])          # shape (3,1)
fila = np.array([[10, 20, 30, 40]])          # shape (1,4)

print("columna shape:", columna.shape)
print("fila shape   :", fila.shape)
print()
resultado = columna * fila
print("resultado shape:", resultado.shape)
print(resultado)

columna shape: (3, 1)
fila shape   : (1, 4)

resultado shape: (3, 4)
[[ 10  20  30  40]
 [ 20  40  60  80]
 [ 30  60  90 120]]


Cómo se leen las reglas, de derecha a izquierda:

| Forma A | Forma B | Resultado | Por qué |
|---|---|---|---|
| `(4,)` | escalar | `(4,)` | el escalar se repite |
| `(2,3)` | `(3,)` | `(2,3)` | la última dimensión coincide (3 y 3) |
| `(3,1)` | `(1,4)` | `(3,4)` | los unos se estiran |
| `(2,3)` | `(2,)` | **error** | 3 ≠ 2 y ninguno es 1 |

Y así se ve el error cuando te equivocas, que te va a pasar:

In [36]:
try:
    tabla + np.array([1.0, 2.0])        # (2,3) con (2,)
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 


> **Cómo leer ese mensaje.** Te está diciendo las dos formas que no pudo alinear. Casi siempre la solución es revisar con `.shape` cuál de los dos arrays no tiene la forma que creías, no pelearse con el *broadcasting*.

### Un caso que sí vas a usar: centrar cada columna

Restarle a cada columna su propio promedio es un uso clásico de *broadcasting*: `(n, 2)` menos `(2,)`.

Aquí aparece `axis` por primera vez. Lo formal es de la semana 7; por ahora quédate con que **`axis=0` recorre las filas, así que opera por columna**.

In [37]:
X = np.array([
    [850.0, 12.5],
    [1200.0, 33.0],
    [430.5, 8.0],
    [2100.0, 51.0],
])

promedios = X.mean(axis=0)        # un promedio POR COLUMNA -> shape (2,)
print("promedios por columna:", np.round(promedios, 2))
print()
centrada = X - promedios          # (4,2) - (2,) -> broadcasting
print("datos centrados:")
print(np.round(centrada, 2))
print()
print("los promedios de lo centrado son ~0:", np.round(centrada.mean(axis=0), 10))

promedios por columna: [1145.12   26.12]

datos centrados:
[[-295.12  -13.62]
 [  54.88    6.88]
 [-714.62  -18.12]
 [ 954.88   24.88]]

los promedios de lo centrado son ~0: [0. 0.]


### `ufuncs`: las funciones que se aplican a todo el array

Una *ufunc* (*universal function*) es una función de NumPy que opera elemento a elemento y devuelve un array. Las que vas a usar en el curso:

In [38]:
a = np.array([1.0, 4.0, 9.0, 16.0, 25.0])

print("sqrt :", np.sqrt(a))
print("log  :", np.round(np.log(a), 3))       # logaritmo natural
print("log10:", np.log10(a))
print("exp  :", np.round(np.exp(np.array([0.0, 1.0, 2.0])), 3))
print()
b = np.array([-3.5, 2.4, -1.6, 0.5])
print("abs  :", np.abs(b))
print("round:", np.round(b))
print("floor:", np.floor(b))
print("ceil :", np.ceil(b))

sqrt : [1. 2. 3. 4. 5.]
log  : [0.    1.386 2.197 2.773 3.219]
log10: [0.         0.60205999 0.95424251 1.20411998 1.39794001]
exp  : [1.    2.718 7.389]

abs  : [3.5 2.4 1.6 0.5]
round: [-4.  2. -2.  0.]
floor: [-4.  2. -2.  0.]
ceil : [-3.  3. -1.  1.]


| `ufunc` | Qué hace | Para qué te sirve aquí |
|---|---|---|
| `np.sqrt` | Raíz cuadrada | Desviación estándar a mano |
| `np.abs` | Valor absoluto | Distancia al promedio, sin signo |
| `np.round` | Redondea | Dejar el JSON de la práctica a 2 decimales |
| `np.log`, `np.log10` | Logaritmos | Aplanar columnas muy sesgadas |
| `np.floor`, `np.ceil` | Hacia abajo / hacia arriba | Agrupar en rangos |
| `np.isnan` | ¿Es `nan`? | Encontrar los valores que no se pudieron convertir |

> **`np.round` redondea al par más cercano** en los empates (`0.5` → `0`, `1.5` → `2`, `2.5` → `2`). No es un bug, es el estándar IEEE 754, el mismo que usa `round()` de Python. Con datos reales casi nunca importa, pero si tus conteos no cuadran por un valor, ya sabes por dónde buscar.

### Cuidado con `log` y `sqrt` sobre datos sucios

Tu columna numérica trae **outliers negativos** a propósito. ¿Qué pasa si le aplicas una raíz o un logaritmo? No truena: te devuelve `nan` y una advertencia que es fácil pasar por alto.

In [39]:
sucio = np.array([100.0, -50.0, 0.0, 25.0])

print("sqrt:", np.sqrt(sucio))     # el negativo -> nan
print("log :", np.log(sucio))      # el negativo -> nan, el cero -> -inf

sqrt: [10. nan  0.  5.]
log : [4.60517019        nan       -inf 3.21887582]


C:\Users\Mario\AppData\Local\Temp\ipykernel_60716\712909031.py:3: RuntimeWarning: invalid value encountered in sqrt
  print("sqrt:", np.sqrt(sucio))     # el negativo -> nan
C:\Users\Mario\AppData\Local\Temp\ipykernel_60716\712909031.py:4: RuntimeWarning: divide by zero encountered in log
  print("log :", np.log(sucio))      # el negativo -> nan, el cero -> -inf
C:\Users\Mario\AppData\Local\Temp\ipykernel_60716\712909031.py:4: RuntimeWarning: invalid value encountered in log
  print("log :", np.log(sucio))      # el negativo -> nan, el cero -> -inf


`nan` = *not a number*; `-inf` = infinito negativo. La advertencia aparece **una sola vez** por sesión, así que si la celda se ejecutó antes ya no la vuelves a ver: es muy fácil quedarte con un array lleno de `nan` creyendo que todo salió bien. Revisa siempre el resultado.

### `nan`: el valor que lo contamina todo

`nan` tiene dos propiedades que hay que conocer de memoria, porque te vas a topar con él en cuanto convierta tu columna sucia:

1. **Cualquier operación con `nan` da `nan`.** Un solo `nan` arruina el promedio del array completo.
2. **`nan` no es igual a nada, ni a sí mismo.** `nan == nan` es `False`. Para detectarlo se usa `np.isnan()`.

In [40]:
print("5 + nan       =", 5 + np.nan)
print("nan == nan    ->", np.nan == np.nan)      # False!
print("np.isnan(nan) ->", np.isnan(np.nan))      # asi SI se detecta

5 + nan       = nan
nan == nan    -> False
np.isnan(nan) -> True


In [41]:
# Una columna con un hueco, como la que vas a sacar de tu CSV
col = np.array([850.0, 1200.0, np.nan, 2100.0, np.nan])

print("col        :", col)
print("cuantos nan:", np.isnan(col).sum())
print()
print("mean()     :", col.mean())          # nan: un hueco contamina todo
print("np.nanmean :", np.nanmean(col))     # ignora los nan

col        : [ 850. 1200.   nan 2100.   nan]
cuantos nan: 2

mean()     : nan
np.nanmean : 1383.3333333333333


Tienes dos estrategias, y las dos son válidas en la P3:

1. **Quedarte solo con los válidos** y calcular normal: `validos = col[~np.isnan(col)]`. Es la que pide la P3 en su paso 1 ("ignorar filas donde el valor sea vacío o no convertible"), y la que usamos el viernes.
2. **Usar las versiones `nan*`**: `np.nanmean`, `np.nanmin`, `np.nanmax`, `np.nanstd`. Útiles cuando necesitas conservar la forma del array.

Lo que **no** es válido es reportar un `nan` como si fuera un resultado.

### Agregaciones: un adelanto

Las operaciones que colapsan un array entero a un solo número. La batería completa (`median`, `percentile`, `std` por eje) es de la semana 7; con estas cuatro basta para hoy.

In [42]:
a = np.array([850.0, 1200.0, 430.5, 2100.0, 675.0])

print("suma  :", a.sum())
print("media :", a.mean())
print("minimo:", a.min())
print("maximo:", a.max())
print("cuenta:", a.size)
print()
# Tambien existen como funcion: np.sum(a) == a.sum()
print("np.sum(a) == a.sum():", np.sum(a) == a.sum())

suma  : 5255.5
media : 1051.1
minimo: 430.5
maximo: 2100.0
cuenta: 5

np.sum(a) == a.sum(): True


### Comparaciones: otro adelanto

Comparar un array con un número devuelve un **array de booleanos**, uno por elemento. Sumar ese array cuenta los `True` (porque `True` vale 1).

In [43]:
a = np.array([850.0, 1200.0, 430.5, 2100.0, 675.0])

grandes = a > 1000
print("a > 1000       ->", grandes)
print("cuantos cumplen:", grandes.sum())
print("que porcentaje :", round(100 * grandes.mean(), 1), "%")

a > 1000       -> [False  True False  True False]
cuantos cumplen: 2
que porcentaje : 40.0 %


Eso que acabas de crear es una **máscara booleana**, y usarla para *filtrar* (`a[grandes]`), combinar condiciones con `&` y `|`, y `np.where` es el tema central de la semana 7 y del paso 3 de la P3. Por ahora basta con que sepas que comparar un array devuelve otro array.

### ¿Por qué es tan rápido?

Tres razones, y entenderlas te dice cuándo NumPy **no** va a ayudarte:

1. **Memoria contigua.** Un array es un bloque único de bytes del mismo tipo. Una lista de Python es un montón de punteros a objetos dispersos en memoria.
2. **Un solo `dtype`.** NumPy no tiene que preguntarse de qué tipo es cada elemento: ya lo sabe.
3. **El bucle corre en C, no en Python.** `.mean()` no hace 100,000 pasos de intérprete; hace uno y deja que código compilado recorra el bloque.

In [44]:
# El mismo calculo, con list comprehension y con NumPy
n = 2_000_000
lista = list(range(n))
arreglo = np.arange(n)

inicio = time.perf_counter()
r1 = [v ** 0.5 for v in lista]
t_py = time.perf_counter() - inicio

inicio = time.perf_counter()
r2 = np.sqrt(arreglo)
t_np = time.perf_counter() - inicio

print(f"list comprehension: {t_py:.4f} s")
print(f"numpy             : {t_np:.4f} s")
print(f"factor            : {t_py / t_np:.0f}x")

list comprehension: 0.1614 s
numpy             : 0.0043 s
factor            : 37x


Y por lo mismo, **cuándo NumPy no es la herramienta**:

| Situación | Por qué no |
|---|---|
| Datos heterogéneos (texto + números + fechas) | El array necesita un solo `dtype`. Eso es trabajo para **pandas**, en la semana 9 |
| Pocos elementos (decenas) | El costo de crear el array se come la ganancia |
| Ir agregando elementos uno por uno | El tamaño es fijo; cada `append` copia todo. Acumula en una lista y convierte **al final** |
| Lógica fila por fila que no se puede expresar como operación de columnas | Un bucle normal es más claro, y más honesto |

Ese último punto importa: si acabas escribiendo `for i in range(len(arr)): arr[i] = ...`, estás usando NumPy sin aprovecharlo, y vas a ser más lento que con una lista.

### Ejercicios sesión 2

In [45]:
# Ejercicio 7: Tienes estos precios sin IVA:
precios = np.array([100.0, 250.5, 80.0, 1200.0])
# a) Calcula los precios con IVA (16%) en una sola operacion.
# b) Redondealos a 2 decimales.
# c) Imprime cuanto suman en total.

# Tu codigo aqui:

In [46]:
# Ejercicio 8: Con estos dos arrays:
monto = np.array([850.0, 1200.0, 430.5, 2100.0, 675.0])
minutos = np.array([12.5, 33.0, 8.0, 51.0, 20.0])
# a) Calcula los pesos por minuto de cada registro.
# b) Imprime el valor mas alto y el mas bajo de esa metrica derivada.

# Tu codigo aqui:

In [47]:
# Ejercicio 9: Esta matriz tiene 4 registros y 3 columnas:
X = np.array([
    [10.0, 100.0, 1000.0],
    [20.0, 200.0, 2000.0],
    [30.0, 300.0, 3000.0],
    [40.0, 400.0, 4000.0],
])
# a) Multiplica la primera columna por 2, la segunda por 10 y la tercera por 100,
#    todo en UNA operacion (pista: broadcasting con un array de 3 elementos).
# b) Resta a cada columna su propio promedio.

# Tu codigo aqui:

In [48]:
# Ejercicio 10: Este array tiene huecos:
col = np.array([45.0, np.nan, 12.0, 88.0, np.nan, 33.0])
# a) Cuantos nan tiene?
# b) Calcula su promedio de las DOS formas vistas (filtrando y con nanmean)
#    y comprueba que dan lo mismo.
# c) Cuantos valores validos quedaron?

# Tu codigo aqui:

In [49]:
# Ejercicio 11: Explica en un comentario por que esto imprime False,
# y luego arreglalo para que detecte correctamente el valor faltante.
x = np.nan
print(x == np.nan)

# Tu codigo aqui:

False


In [50]:
# Ejercicio 12 (reto): Sin usar np.std, calcula la desviacion estandar de
# este array usando solo mean, resta, ** y sqrt.
a = np.array([850.0, 1200.0, 430.5, 2100.0, 675.0])
# La formula es: raiz del promedio de las diferencias al cuadrado respecto a la media.
# Comprueba tu resultado contra np.std(a).

# Tu codigo aqui:

## Sesión 3 — Ejercicio integrador

Hoy juntamos todo sobre un CSV con la misma forma que el tuyo: separado por `|`, con celdas vacías y valores basura.

**Lo que vamos a construir es, literalmente, la P3:**

| Paso de hoy | Paso de la P3 |
|---|---|
| Leer el CSV y sacar una columna | — |
| Convertir a float marcando lo inválido | Paso 1: "cargar las columnas numéricas ignorando lo no convertible" |
| Inspeccionar `shape`, `dtype`, `nan` | Paso 1 (`n_valores_validos`) |
| Estadísticas sobre los válidos | Paso 2 (ahí con `median` y `percentile`, de la semana 7) |
| Las dos numéricas en una matriz | Paso 1 con las dos columnas |
| Benchmark bucle vs NumPy | **Paso 4** |

Lo único de la P3 que hoy **no** vemos es la detección de outliers con máscaras (paso 3): eso es la semana 7.

### Los datos de ejemplo

Esta celda escribe un CSV de juguete con los mismos problemas que tu dataset: separador `|`, celdas vacías y un valor no numérico. Es el mismo archivo que usamos la semana pasada, para que puedas comparar lo que te costó con Python puro.

In [51]:
%%writefile ventas_ejemplo.csv
producto|monto_venta|minutos_en_sitio|email|comentario_cliente
Laptop|15000.50|45.0|ana@gmail.com|Excelente equipo, llego a tiempo
Mouse|350|12.0|luis@hotmail.com|Bueno
Laptop|14500||carlosgmail.com|Caro pero vale la pena
Teclado|800|8.5||Sin comentarios
Monitor|5000|33.0|sofia@yahoo.com|
Laptop||51.0|diego@live.com|Muy satisfecho con la compra
Mouse|400|5.0|val@outlook.com|Cumple
Impresora|3500|27.5|andrea@gmail.com|Tarda en imprimir, Calle Juarez #12, CDMX
Mouse|320||miguel@gmail.com|Regular
Laptop|16000|62.0|paula@gmail.com|La recomiendo
Teclado|750|9.0|jose@gmail.com|Buena relacion precio-calidad
Monitor|abc|41.0|rosa@gmail.com|Dato capturado mal

Writing ventas_ejemplo.csv


### Paso 1 — Leer el archivo y localizar la columna

NumPy no es bueno leyendo CSVs sucios y con texto. Lo que sí funciona —y es lo que pide la P3— es leer con `open()`, separar con `.split("|")`, y usar NumPy a partir de que ya tengas números.

Fíjate en que la columna se localiza **por su nombre**, con `.index()`, nunca con un número escrito a mano: el orden de las columnas de tu dataset es aleatorio según tu semilla.

In [52]:
with open("ventas_ejemplo.csv", "r", encoding="utf-8") as f:
    lineas = f.read().strip().split("\n")

cabecera = lineas[0].split("|")
filas = [linea.split("|") for linea in lineas[1:]]

print("columnas:", cabecera)
print("filas de datos:", len(filas))
print()

# Localizar la columna POR NOMBRE, nunca por un indice escrito a mano
i_monto = cabecera.index("monto_venta")
i_minutos = cabecera.index("minutos_en_sitio")
print("monto_venta esta en la posicion", i_monto)
print("minutos_en_sitio esta en la posicion", i_minutos)
print()

crudos = [fila[i_monto] for fila in filas]
print("valores crudos:", crudos)

columnas: ['producto', 'monto_venta', 'minutos_en_sitio', 'email', 'comentario_cliente']
filas de datos: 12

monto_venta esta en la posicion 1
minutos_en_sitio esta en la posicion 2

valores crudos: ['15000.50', '350', '14500', '800', '5000', '', '400', '3500', '320', '16000', '750', 'abc']


Ahí están los dos problemas a la vista: una cadena vacía `''` (celda faltante) y un `'abc'` (dato mal capturado). En tu archivo real hay ~4,000 de las primeras.

### Paso 2 — Convertir a float sin que truene

Ya viste que `astype(float)` se cae con un solo valor malo. La solución es una función que intente convertir y devuelva `np.nan` cuando no pueda.

In [53]:
def a_float(texto):
    """Convierte texto a float; devuelve np.nan si no se puede."""
    try:
        return float(texto.strip())
    except (ValueError, AttributeError):
        return np.nan


# Probarla con los casos que importan
for v in ["850.5", "  1200  ", "", "abc", "-50"]:
    print(repr(v), "->", a_float(v))

'850.5' -> 850.5
'  1200  ' -> 1200.0
'' -> nan
'abc' -> nan
'-50' -> -50.0


In [54]:
# Ahora si: construir el array
monto = np.array([a_float(v) for v in crudos])

print(monto)
print()
print("dtype:", monto.dtype)
print("shape:", monto.shape)

[15000.5   350.  14500.    800.   5000.      nan   400.   3500.    320.
 16000.    750.      nan]

dtype: float64
shape: (12,)


> **¿Por qué `np.nan` y no simplemente saltarse los valores malos?** Porque así el array **conserva la correspondencia con las filas**: la posición 5 del array sigue siendo la fila 5 del CSV. Eso te permite después cruzar dos columnas entre sí, como vamos a hacer en el paso 5. Si vas descartando valores sobre la marcha, los arrays de dos columnas quedan de distinto tamaño y ya no se pueden comparar.
>
> Y nota que `a_float` es una función de **Python puro**, no de NumPy: el patrón general es limpiar con Python lo que viene como texto, y pasar a NumPy en cuanto tengas números.

### Paso 3 — Inspeccionar antes de calcular

Tres líneas que deberías correr siempre después de construir un array a partir de datos sucios.

In [55]:
n_total = monto.size
n_nan = np.isnan(monto).sum()
n_validos = n_total - n_nan

print("valores en total :", n_total)
print("no convertibles  :", n_nan)
print("validos          :", n_validos)
print("porcentaje valido:", round(100 * n_validos / n_total, 2), "%")

valores en total : 12
no convertibles  : 2
validos          : 10
porcentaje valido: 83.33 %


Ese `n_validos` es exactamente el campo `n_valores_validos` que la P3 te pide reportar en su JSON, para cada una de tus dos columnas numéricas.

### Paso 4 — Quedarte con los válidos y calcular

`~np.isnan(monto)` es "los que **no** son nan" (`~` niega la máscara). Usarla como índice devuelve solo esos valores. Esto es la máscara booleana de la semana 7, aquí en su uso más simple.

In [56]:
validos = monto[~np.isnan(monto)]

print("validos:", validos)
print("cuantos:", validos.size)
print()
print("minimo :", validos.min())
print("maximo :", validos.max())
print("media  :", round(validos.mean(), 2))
print("suma   :", validos.sum())

validos: [15000.5   350.  14500.    800.   5000.    400.   3500.    320.  16000.
   750. ]
cuantos: 10

minimo : 320.0
maximo : 16000.0
media  : 5662.05
suma   : 56620.5


In [57]:
# La ruta alternativa: funciones nan*, sin filtrar
print("nanmin :", np.nanmin(monto))
print("nanmax :", np.nanmax(monto))
print("nanmean:", round(np.nanmean(monto), 2))
print()
print("dan lo mismo?", np.allclose(
    [validos.min(), validos.max(), validos.mean()],
    [np.nanmin(monto), np.nanmax(monto), np.nanmean(monto)],
))

nanmin : 320.0
nanmax : 16000.0
nanmean: 5662.05

dan lo mismo? True


### Paso 5 — Las dos columnas numéricas en una matriz

La P3 trabaja con tus **dos** columnas numéricas. Juntarlas en un array 2-D te deja operar sobre las dos a la vez; `np.column_stack` las apila como columnas.

In [58]:
minutos = np.array([a_float(fila[i_minutos]) for fila in filas])

X = np.column_stack([monto, minutos])

print("shape:", X.shape)        # (12, 2): 12 registros, 2 variables
print()
print(X)

shape: (12, 2)

[[1.50005e+04 4.50000e+01]
 [3.50000e+02 1.20000e+01]
 [1.45000e+04         nan]
 [8.00000e+02 8.50000e+00]
 [5.00000e+03 3.30000e+01]
 [        nan 5.10000e+01]
 [4.00000e+02 5.00000e+00]
 [3.50000e+03 2.75000e+01]
 [3.20000e+02         nan]
 [1.60000e+04 6.20000e+01]
 [7.50000e+02 9.00000e+00]
 [        nan 4.10000e+01]]


In [59]:
# Promedio de cada columna, ignorando nan (axis=0 = por columna)
print("promedios por columna:", np.round(np.nanmean(X, axis=0), 2))
print("minimos  por columna:", np.nanmin(X, axis=0))
print("maximos  por columna:", np.nanmax(X, axis=0))
print()
# Y una columna suelta sigue siendo X[:, i]
print("columna monto  :", X[:, 0])
print("columna minutos:", X[:, 1])

promedios por columna: [5662.05   29.4 ]
minimos  por columna: [320.   5.]
maximos  por columna: [16000.    62.]

columna monto  : [15000.5   350.  14500.    800.   5000.      nan   400.   3500.    320.
 16000.    750.      nan]
columna minutos: [45.  12.   nan  8.5 33.  51.   5.  27.5  nan 62.   9.  41. ]


### Paso 6 — Una variable derivada, vectorizada

Pesos por minuto en el sitio: una división de columna entre columna. Donde cualquiera de las dos sea `nan`, el resultado es `nan`, que es el comportamiento correcto —no hay dato para calcularlo.

In [60]:
pesos_por_minuto = X[:, 0] / X[:, 1]

print("por registro:", np.round(pesos_por_minuto, 2))
print()
print("promedio de la metrica:", round(np.nanmean(pesos_por_minuto), 2))
print("registro mas alto     :", round(np.nanmax(pesos_por_minuto), 2))

por registro: [333.34  29.17    nan  94.12 151.52    nan  80.   127.27    nan 258.06
  83.33    nan]

promedio de la metrica: 144.6
registro mas alto     : 333.34


> **Ojo con dividir entre cero.** Si la columna de abajo trae ceros, NumPy no truena: devuelve `inf` (o `nan` si arriba también hay cero) y te avisa con un `RuntimeWarning`. En tu dataset puede pasar, así que revisa el resultado.

In [61]:
a = np.array([100.0, 50.0, 0.0])
b = np.array([5.0, 0.0, 0.0])
print(a / b)       # 20, inf, nan

[20. inf nan]


C:\Users\Mario\AppData\Local\Temp\ipykernel_60716\2552453601.py:3: RuntimeWarning: divide by zero encountered in divide
  print(a / b)       # 20, inf, nan
C:\Users\Mario\AppData\Local\Temp\ipykernel_60716\2552453601.py:3: RuntimeWarning: invalid value encountered in divide
  print(a / b)       # 20, inf, nan


### Paso 7 — El benchmark que pide la P3

El paso 4 de la P3 te pide comparar un bucle de Python puro contra NumPy, con dos reglas que vale la pena leer dos veces:

1. **Sobre la lista de valores ya filtrados**, no sobre el archivo crudo.
2. **Sin incluir el tiempo de leer y parsear el CSV.** Solo el cálculo.

Con 12 valores los tiempos son ruido, así que para que el benchmark signifique algo lo hacemos con 100,000 valores (del tamaño de tu dataset real).

In [62]:
# Simular una columna del tamanio de la tuya, ya filtrada
rng = np.random.default_rng(42)          # semilla fija: reproducible
valores_np = rng.normal(850, 180, 100_000)
valores_py = valores_np.tolist()         # la misma data, como lista de Python

print("cuantos valores:", len(valores_py))

cuantos valores: 100000


In [63]:
# --- Bucle de Python puro: media y desviacion estandar ---
inicio = time.perf_counter()

n = len(valores_py)
suma = 0.0
for v in valores_py:
    suma += v
media_py = suma / n

acum = 0.0
for v in valores_py:
    acum += (v - media_py) ** 2
std_py = (acum / n) ** 0.5

t_bucle = time.perf_counter() - inicio

# --- NumPy ---
inicio = time.perf_counter()
media_np = valores_np.mean()
std_np = valores_np.std()
t_numpy = time.perf_counter() - inicio

print(f"bucle -> media {media_py:.4f} | std {std_py:.4f} | {t_bucle:.6f} s")
print(f"numpy -> media {media_np:.4f} | std {std_np:.4f} | {t_numpy:.6f} s")
print()
print("mismo resultado?", np.allclose([media_py, std_py], [media_np, std_np]))
print(f"factor de velocidad: {t_bucle / t_numpy:.1f}x")

bucle -> media 849.2381 | std 180.6666 | 0.014470 s


numpy -> media 849.2381 | std 180.6666 | 0.001001 s

mismo resultado? True
factor de velocidad: 14.4x


Los dos dan el mismo número —compruébalo siempre con `np.allclose`, porque un benchmark donde las dos versiones calculan cosas distintas no mide nada.

### El detalle del reloj: por qué `perf_counter` y no `time`

Este benchmark se escribió primero con `time.time()`, como dice el enunciado de la P3, **y truena**. Esto es lo que pasa:

In [64]:
# Lo mismo, pero midiendo con time.time()
inicio = time.time()
valores_np.mean()
valores_np.std()
t_malo = time.time() - inicio

print("medido con time.time()      :", t_malo, "s")

inicio = time.perf_counter()
valores_np.mean()
valores_np.std()
t_bueno = time.perf_counter() - inicio

print("medido con time.perf_counter():", t_bueno, "s")
print()
try:
    print("factor con time.time():", t_bucle / t_malo)
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e, "<- el reloj no alcanza a ver nada")

medido con time.time()      : 0.0010020732879638672 s
medido con time.perf_counter(): 0.0002538999542593956 s

factor con time.time(): 14.440061646814776


En Windows, `time.time()` avanza a saltos de ~16 ms. El cálculo de NumPy sobre 100,000 valores tarda menos que eso, así que las dos lecturas del reloj salen **idénticas** y la resta da exactamente `0.0`. Después, al calcular `tiempo_bucle / tiempo_numpy`, te truena con `ZeroDivisionError`.

Tienes dos salidas, y las dos son aceptables en la P3:

1. **Medir con `time.perf_counter()`** (recomendado). Se usa igual que `time.time()`, pero está hecho para medir intervalos y tiene resolución de nanosegundos.
2. **Repetir la operación** 100 o 1,000 veces dentro de la medición y dividir el total entre el número de repeticiones, para que el tiempo acumulado sí sea medible.

> **Lo que tu JSON de la P3 debe reportar en `benchmark`:** `n_filas` (que tiene que coincidir con el `n_valores_validos` de tu `<numerica_1>`), `tiempo_bucle_python_seg`, `tiempo_numpy_seg`, `factor_velocidad` y `operacion_medida`. Dos señales de que algo salió mal: un `tiempo_numpy_seg` de `0.0` (problema del reloj, el de arriba) y un factor muy chico, de menos de 10x (casi siempre es porque metiste la lectura del CSV dentro de la medición).

### Qué te llevas de hoy

El patrón completo, que es el esqueleto del paso 1 de tu P3:

```python
# 1. Python puro: leer y localizar la columna por nombre
filas = [linea.split("|") for linea in lineas[1:]]
i = cabecera.index("mi_columna_numerica")

# 2. Python puro: convertir marcando lo invalido
col = np.array([a_float(fila[i]) for fila in filas])

# 3. NumPy: quedarte con los validos
validos = col[~np.isnan(col)]

# 4. NumPy: calcular
print(validos.size, validos.mean(), validos.min(), validos.max())
```

Lo que falta para completar la P3 —`median`, `percentile`, `std`, outliers a ±3 desviaciones y máscaras booleanas— es la **semana 7**.

### Ejercicios sesión 3

In [65]:
# Ejercicio 13: Repite los pasos 1 a 4 pero con la columna minutos_en_sitio:
# localizala por nombre, conviertela, cuenta cuantos nan tiene y
# reporta n_validos, minimo, maximo y media de los validos.

# Tu codigo aqui:

In [66]:
# Ejercicio 14: Cuantos registros tienen las DOS columnas numericas validas
# al mismo tiempo? (pista: ~np.isnan(monto) y ~np.isnan(minutos),
# combinadas con &). Imprime el numero y el porcentaje.

# Tu codigo aqui:

In [67]:
# Ejercicio 15: Arma el diccionario que la P3 pide para numerica_1, con las
# claves que ya puedes calcular esta semana, y redondeado a 2 decimales:
#   {"columna": ..., "n_valores_validos": ..., "media": ...,
#    "minimo": ..., "maximo": ...}
# Imprimelo con json.dumps(..., indent=2, ensure_ascii=False).
# Ojo: json.dump no sabe escribir tipos de NumPy; convierte con float() o int().

# Tu codigo aqui:

In [68]:
# Ejercicio 16 (reto): Escribe una funcion
#   cargar_columna(ruta, nombre_columna, separador="|")
# que devuelva una tupla (array_completo_con_nan, array_solo_validos).
# Debe localizar la columna por nombre y funcionar con cualquiera de las
# columnas numericas del archivo. Pruebala con las dos.
# Esta funcion es, tal cual, la que te va a servir en la P3.

# Tu codigo aqui:

---

## Resumen de la semana

**Crear arrays**

| Qué quieres | Cómo |
|---|---|
| Desde una lista | `np.array([1, 2, 3])` |
| Un rango con paso | `np.arange(0, 10, 2)` |
| N valores entre dos límites | `np.linspace(0, 1, 11)` |
| Relleno | `np.zeros(5)`, `np.ones(3)`, `np.full(4, 7.5)` |
| 2-D | `np.array([[1, 2], [3, 4]])`, `np.zeros((2, 3))` |
| Apilar columnas | `np.column_stack([a, b])` |
| Cambiar de forma | `a.reshape(3, 4)`, `a.reshape(-1, 2)` |

**Inspeccionar** — los cuatro que revisas cuando algo falla

`a.shape` (forma, siempre tupla) · `a.ndim` (dimensiones) · `a.size` (total de elementos) · `a.dtype` (tipo)

**Indexar**

| Qué | Cómo |
|---|---|
| Un elemento (1-D / 2-D) | `a[3]` / `a[1, 2]` |
| Una fila | `t[2]` |
| **Una columna** | `t[:, 1]` |
| Un bloque | `t[0:3, 1:4]` |
| Al revés | `a[::-1]` |
| Posiciones sueltas | `a[[0, 2, 4]]` |
| **Copia independiente** | `a[0:3].copy()` |

**Operar**

- Con un escalar o entre arrays: `a + 10`, `a * b`, `a / b` — siempre elemento a elemento
- *Broadcasting*: las formas se comparan de derecha a izquierda; compatibles si son iguales o si una es 1
- `ufuncs`: `np.sqrt`, `np.abs`, `np.round`, `np.log`, `np.isnan`
- Agregaciones: `a.sum()`, `a.mean()`, `a.min()`, `a.max()`; `axis=0` opera **por columna**
- `nan`: contamina todo, y `nan == nan` es `False`. Se detecta con `np.isnan`, se ignora con `np.nanmean` o filtrando con `a[~np.isnan(a)]`
- Comparar devuelve booleanos: `(a > 1000).sum()` cuenta cuántos cumplen

**El patrón del curso**

Python puro para leer y limpiar el texto → `np.array` en cuanto tengas números → NumPy para todo el cálculo.

## Las cinco trampas de esta semana

1. **Meter strings a `np.array` sin convertir.** El `dtype` queda en `<U...` y nada matemático funciona. Revisa siempre `.dtype`.
2. **`astype(float)` sobre una columna sucia.** Un solo `""` o `"abc"` tumba la conversión completa. Convierte valor por valor con una función que devuelva `np.nan`.
3. **Creer que una rebanada es una copia.** `b = a[0:3]` es una **vista**: si modificas `b`, cambias `a`. Usa `.copy()` cuando importe.
4. **Reportar un `nan` como resultado.** Si tu promedio salió `nan`, no es el promedio: te falta filtrar o usar `np.nanmean`.
5. **Escribir un `for` sobre un array.** Si estás recorriendo índices para modificar valores uno por uno, perdiste todo lo que NumPy te da. Busca la operación vectorizada.

## Qué sigue

**Semana 7 — NumPy II: estadística y filtrado**, donde se cierra todo lo que la P3 necesita:

- Estadística vectorizada: `mean`, `std`, `median`, `percentile`, y operaciones por eje (`axis=0`, `axis=1`)
- **Máscaras booleanas** para filtrar de verdad: `a[a > umbral]`, combinar condiciones con `&`, `|`, `~`, y `np.where`
- Detección de outliers a ±3 desviaciones estándar — el paso 3 de la P3
- Benchmarking formal de NumPy contra bucles

Y ese viernes es el **Examen 1** (terminal, entornos virtuales, Git/GitHub y Python I-III).

> **Para llegar listo:** vuelve a correr el ejercicio integrador de hoy, pero sobre **tu** archivo `datos/{tema}-ruido.csv` y con **tus** dos columnas numéricas. Si eso te sale, la P3 es ponerle encima la estadística de la semana 7.